# Part 5 — Mixture-of-Experts: A Walkthrough

## Think of it as a medical clinic 🏥

Before any code, here's the one picture that explains this entire notebook. Keep it in your head the whole way through:

> A clinic has **several specialist doctors** (a heart doctor, a skin doctor, a bone doctor...). When a patient walks in, a **receptionist** reads their symptoms and sends them to the **2 most relevant** doctors — not all of them. Each doctor only sees the patients that suit them, so they get really good at their specialty. The clinic *knows* a lot (many specialists on staff), but each visit is *cheap* (you only take up 2 doctors' time).

That's a Mixture-of-Experts layer. Swap a few words and you have the real thing:

| Clinic | MoE layer |
|---|---|
| patient | a **token** (one word's vector) |
| specialist doctor | an **expert** (a small neural network) |
| receptionist who picks doctors | the **router** (a tiny layer that scores experts) |
| "see your 2 best-matched doctors" | **top-k routing** (k = 2) |
| clinic manager keeping work fair | the **load-balancing loss** |

**Why this part exists:** in Parts 1 and 3 every word went through *one* feed-forward network — like a clinic with a single overworked general doctor who must handle everything. MoE replaces that one doctor with a *team of specialists plus a receptionist*. The model can hold far more knowledge without each word costing more to process.

**What you should know going in:**
- Part 1 (the plain `Linear -> GELU -> Linear` feed-forward) and Part 3 (SwiGLU) are fresh
- You've seen `softmax` (turns scores into percentages) and `top-k` (pick the largest few)

**What you'll build, step by step:**
1. **5.1** the receptionist — how the router picks doctors for each word
2. **5.2** keeping it fair — why we must stop everyone from piling onto one doctor
3. **5.3** the doctors themselves — each expert is just a Part 3 SwiGLU
4. **5.4** the full visit — send each word to its doctors and combine their answers
5. **5.5** adding a general doctor on top (the hybrid block)
6. **5.6** dropping the whole clinic into a real Transformer block

**One thing to keep clear:** this part is *only* the feed-forward piece. No attention, no training loop — it's a drop-in replacement for the feed-forward sublayer you already know.

We follow one tiny sentence — `"I love deep learning"` (4 words) — through every step, with small numbers so everything fits on screen.


## The Map

Here's the whole clinic visit as a flowchart. You'll see this same diagram at the top of every section, with the step we're on lit up in green and the rest greyed out — so you always know where you are.

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#e8e8e8,stroke:#bbb,color:#777
    style aux fill:#e8e8e8,stroke:#bbb,color:#777
    style exp fill:#e8e8e8,stroke:#bbb,color:#777
    style comb fill:#e8e8e8,stroke:#bbb,color:#777
    style hyb fill:#e8e8e8,stroke:#bbb,color:#777
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

Reading it as a clinic visit:
1. a word arrives (**input**) and we line all the words up (**flatten**)
2. the **receptionist** (5.1) scores the doctors and picks the best 2 for each word
3. a side-check makes sure the workload stays **fair** (5.2)
4. the chosen **doctors** (5.3) each examine the word
5. we **combine** their answers, weighted by how strongly the receptionist recommended each (5.4)
6. optionally blend in a general doctor too (**hybrid**, 5.5), then hand the result onward

**Where this sits in the bigger model:** the feed-forward step has now been done three different ways across the course. This is just the third version of the *same slot*:

| Part | The feed-forward step is... | In clinic terms |
|---|---|---|
| 1 | `Linear -> GELU -> Linear` | one general doctor |
| 3 | SwiGLU (a smarter single network) | one *better* general doctor |
| **5** | **MoE: receptionist + several specialists** | **a whole clinic** |

**Don't mix up the two "gates"** (the word gets reused):
- Part 3's SwiGLU has an *internal* gate that decides how much of each feature to keep — but every word still goes through that one network.
- Part 5's **router** decides something different: *which networks even run* for this word.

**The single most important idea in this part:** you can put **many** specialists on staff (lots of knowledge) while each word only visits **2** of them (cheap per word). More experts → more total knowledge, *without* making each word more expensive. That trade is the entire reason MoE exists, and it's how models like Mixtral pack huge capacity into an affordable per-word cost.

**Our tiny example everywhere:** the sentence `"I love deep learning"` = **4 words**, each described by **8 numbers** (`C = 8`), a clinic of **4 doctors** (`E = 4`), and every word sees its **2 best** (`k = 2`).


## Setup

Run this cell once. It adds `part_5/` (local modules) and `part_3/` (for the block integration in section 5.6) to the path.


In [ ]:
import sys, pathlib

# This notebook lives inside part_5/. Add it and part_3/ (for section 5.6) to sys.path.
NB_DIR = pathlib.Path().resolve()
PART_3 = NB_DIR.parent / "part_3"
for p in (str(NB_DIR), str(PART_3)):
    if p not in sys.path:
        sys.path.insert(0, p)

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

print("Setup complete. Working directory:", NB_DIR)


### Our four "patients"

Same setup as Parts 1-4: imagine the four words *I*, *love*, *deep*, *learning* have already been turned into vectors of `C = 8` numbers each (in a real model these come out of the attention step just before). These four word-vectors are the "patients" walking into our clinic. We fix the random seed so every section sees the exact same four patients.


In [ ]:
# Anchor sentence: "I love deep learning"  ->  T = 4 tokens
TOKENS = ["I", "love", "deep", "learning"]
T = len(TOKENS)

# Tiny MoE hyper-parameters used throughout the notebook
B = 1
C = 8          # d_model
E = 4          # n_expert
K = 2          # top-k experts per token
S = B * T      # flattened token count

torch.manual_seed(42)
x_anchor = torch.randn(B, T, C)
x_flat = x_anchor.reshape(S, C)   # routing is per token -> flatten batch and time

print("x_anchor:", tuple(x_anchor.shape), " -> x_flat:", tuple(x_flat.shape))
print(f"dimensions: B={B}, T={T}, C={C}, E={E}, k={K}")


---
## 5.1 The Receptionist (the Router)

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style aux fill:#e8e8e8,stroke:#bbb,color:#777
    style exp fill:#e8e8e8,stroke:#bbb,color:#777
    style comb fill:#e8e8e8,stroke:#bbb,color:#777
    style hyb fill:#e8e8e8,stroke:#bbb,color:#777
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

### What we're doing here, and why

A word walks in. **Someone has to decide which doctors it should see.** That someone is the *router* — our receptionist. It's tiny: it looks at the word and gives each of the 4 doctors a score, then picks the 2 highest.

Mechanically it does three quick things:
1. **Score** every doctor — a small layer (`Linear`) turns the word's 8 numbers into 4 scores, one per doctor.
2. **Turn scores into percentages** — `softmax` squashes those 4 scores so they add up to 100% ("60% heart, 30% skin, 7% bone, 3% other").
3. **Pick the top 2** — `topk` keeps the two highest and remembers how confident the receptionist was about each.

Why a layer that *learns* instead of a fixed rule? Because the receptionist trains alongside the doctors. As the doctors get good at certain words, the receptionist learns to send those words their way — the two improve together. And words that look alike (similar vectors) get sent to the same doctors, which is exactly what lets each doctor build up a specialty.

### The formula behind it (optional)

$$\text{scores} = x W_g + b \qquad p = \text{softmax}(\text{scores}) \qquad (\text{weights}, \text{ids}) = \text{topk}(p, k)$$

**One detail worth knowing:** the weights we keep are the raw percentages from `softmax` — we do **not** rescale the chosen 2 to add back up to 100%. So a hesitant receptionist ("maybe 40% this doctor, 20% that one") automatically dials down how much those doctors' answers count. That's a deliberate choice (some other implementations rescale instead); just know the weights per word sum to *at most* 1, not exactly 1.

### Let's watch the receptionist score our four words


In [ ]:
from gating import TopKGate

torch.manual_seed(7)
gate = TopKGate(dim=C, n_expert=E, k=K)

# Peek inside: do each step of the gate manually
with torch.no_grad():
    logits = gate.w_g(x_flat)                 # (S, E)
    probs  = torch.softmax(logits, dim=-1)    # (S, E)

print("Router logits (S, E):")
for t, tok in enumerate(TOKENS):
    print(f"  {tok:9s} {logits[t].numpy()}")
print()
print("Routing probabilities (each row sums to 1):")
for t, tok in enumerate(TOKENS):
    print(f"  {tok:9s} {probs[t].numpy()}   sum={probs[t].sum():.3f}")


In [ ]:
# Now the real gate call: top-k selection + aux loss
with torch.no_grad():
    idx, w, aux = gate(x_flat)

print("Top-k routing decisions (k=2, slot 0 = primary expert):")
print(f"  {'token':9s} {'experts':12s} {'weights':22s} sum(w)")
for t, tok in enumerate(TOKENS):
    print(f"  {tok:9s} {idx[t].tolist()!s:12s} {[round(v,3) for v in w[t].tolist()]!s:22s} {w[t].sum():.3f}  (<= 1, not renormalized)")
print(f"\naux loss: {aux.item():.4f}  (explained in section 5.2)")


### Picture it — who each word gets sent to

Each row is a word, each column a doctor. Darker green = the receptionist liked that doctor more for that word. The red boxes are the 2 doctors actually chosen (thick box = the top pick).


In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(6, 3.2))
im = ax.imshow(probs.numpy(), cmap="YlGn", vmin=0, vmax=probs.max().item())
ax.set_xticks(range(E)); ax.set_xticklabels([f"Expert {e}" for e in range(E)])
ax.set_yticks(range(S)); ax.set_yticklabels(TOKENS)
ax.set_title(f"Router probabilities — red outline = chosen top-{K}")
for t in range(S):
    for e in range(E):
        ax.text(e, t, f"{probs[t,e].item():.2f}", ha="center", va="center", fontsize=9)
    for slot in range(K):
        e = idx[t, slot].item()
        lw = 2.5 if slot == 0 else 1.2          # primary gets a thicker outline
        ax.add_patch(mpatches.Rectangle((e-0.5, t-0.5), 1, 1, fill=False, edgecolor="red", lw=lw))
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


### Let's do the receptionist's math by hand

Nothing here is magic — let's prove it. Take the first word (*I*). The receptionist gave it 4 raw scores. To turn those into percentages, `softmax` just exponentiates each score and divides by the total:

$$\text{percentage for doctor } e = \frac{e^{\text{score}_e}}{e^{\text{score}_0} + e^{\text{score}_1} + e^{\text{score}_2} + e^{\text{score}_3}}$$

We compute that ourselves below and check it matches PyTorch's `softmax` exactly — then confirm the 2 chosen doctors really are the 2 with the highest percentages.


In [ ]:
l0 = logits[0]                                   # logits for token "I"
p0_manual = torch.exp(l0) / torch.exp(l0).sum()  # softmax by hand

print("logits for 'I'        :", l0.numpy())
print("softmax by hand       :", p0_manual.numpy())
print("softmax from torch    :", probs[0].numpy())
print("max abs diff          :", (p0_manual - probs[0]).abs().max().item())
print()
order = torch.argsort(p0_manual, descending=True)
print(f"two largest probs sit at experts {order[:2].tolist()}  ->  gate picked {idx[0].tolist()}")


### Shape trace

| Stage | Shape |
|---|---|
| input `x_flat` | `(S, C)` = `(4, 8)` |
| `logits = w_g(x)` | `(S, E)` = `(4, 4)` |
| `probs = softmax(logits)` | `(4, 4)` — rows sum to 1 |
| `idx` (expert ids) | `(S, k)` = `(4, 2)` long |
| `w` (gate weights) | `(4, 2)` float — per-row sum ≤ 1 |
| `aux` | scalar |

### Why this works (from simple to deep)

- **Simple:** it's the cheapest possible decision-maker — just one small scoring layer and a softmax. The receptionist is almost free compared to the doctors.
- **Practical:** the receptionist *learns on the job*. Its scores feed into the final answer, so training nudges it to make better referrals over time — the receptionist and the doctors get better together, not separately.
- **Deeper:** similar words get similar scores, so they're sent to the same doctors. Over training, each doctor ends up seeing one "kind" of word again and again — which is exactly how a specialist beats a generalist *on its specialty*. A general doctor who must handle everything can't go as deep on any one thing.


---
## 5.2 Keeping the Workload Fair

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#e8e8e8,stroke:#bbb,color:#777
    style aux fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style exp fill:#e8e8e8,stroke:#bbb,color:#777
    style comb fill:#e8e8e8,stroke:#bbb,color:#777
    style hyb fill:#e8e8e8,stroke:#bbb,color:#777
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

### What we're doing here, and why

Our clinic has a dangerous failure mode. Imagine one doctor is, by pure luck, slightly better on day one. The receptionist notices and sends them a few more patients. With more practice that doctor gets even better, so the receptionist sends them *even more*... until one doctor sees **every** patient and the other three sit idle, slowly forgetting their skills. Now you're paying four salaries for one working doctor. This is called **router collapse**, and it's the number-one thing that breaks MoE.

The fix is exactly what a good clinic manager does: **add a gentle rule that rewards spreading the work evenly.** We turn that rule into a small extra penalty (an "auxiliary loss") that we add during training. When work is lopsided the penalty is large; when it's spread evenly the penalty is at its smallest. Training, trying to make the total penalty small, naturally pushes the receptionist to share the load.

### How we measure "fairness"

For each doctor we track two simple numbers across a batch of words:

- **importance** = the *average confidence* the receptionist had in this doctor (averaged over all words). Think: "how much did the receptionist *lean toward* this doctor overall?"
- **load** = the *fraction of words* whose **top pick** was this doctor. Think: "what share of patients actually landed here?"

Then the fairness penalty multiplies the two, doctor by doctor, and sums:

$$\text{penalty} = E \cdot \sum_{\text{doctors}} (\text{importance} \times \text{load})$$

The `× E` (number of doctors) is just a scaling so the numbers come out tidy. The penalty is big only when a doctor is high on **both** counts — i.e. when one doctor hogs both the confidence and the patients. (Training detail: only *importance* can be nudged by gradients; *load* comes from a hard count. So importance acts as the steerable stand-in that training pushes around to even things out.)

### Two extremes, worked out by hand

**Perfectly fair** — every doctor gets an equal `1/4` share of both confidence and patients:

$$\text{penalty} = 4 \cdot \big(4 \times \tfrac{1}{4} \times \tfrac{1}{4}\big) = 1.0 \quad \text{(the best you can do)}$$

**Fully collapsed** — one doctor has all the confidence and all the patients (≈1), the rest ≈0:

$$\text{penalty} \approx 4 \cdot (1 \times 1) = 4.0 \quad \text{(the worst case = number of doctors)}$$

So the penalty always sits between **1 (perfectly fair)** and **E (total collapse)**. Let's confirm both numbers in code:


In [ ]:
def aux_loss(probs, top1):
    """Same formula as TopKGate: E * sum(importance * load)."""
    S, E_ = probs.shape
    importance = probs.mean(dim=0)
    load = torch.bincount(top1, minlength=E_).float() / S
    return E_ * (importance * load).sum(), importance, load

S_demo = 8

# Scenario A: perfectly balanced — uniform probs, tokens spread evenly
probs_bal = torch.full((S_demo, E), 1.0 / E)
top1_bal  = torch.arange(S_demo) % E
aux_bal, imp_bal, load_bal = aux_loss(probs_bal, top1_bal)

# Scenario B: fully collapsed — all mass on expert 2
probs_col = torch.full((S_demo, E), 0.005)
probs_col[:, 2] = 1.0 - 0.005 * (E - 1)
top1_col  = torch.full((S_demo,), 2, dtype=torch.long)
aux_col, imp_col, load_col = aux_loss(probs_col, top1_col)

print(f"balanced : aux = {aux_bal.item():.4f}   (hand-derived: 1.0)")
print(f"collapsed: aux = {aux_col.item():.4f}   (hand-derived: ~{E}.0)")


### Picture it — a fair clinic vs a collapsed one

Left: every doctor gets a roughly equal share (penalty near 1). Right: one doctor hogs everything (penalty near 4). Blue bars = confidence (importance), orange bars = patient share (load), dashed line = the fair target of 1/4 each.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, imp, load, aux_v, title in [
    (axes[0], imp_bal, load_bal, aux_bal, "Balanced router"),
    (axes[1], imp_col, load_col, aux_col, "Collapsed router"),
]:
    xs = np.arange(E)
    ax.bar(xs - 0.18, imp.numpy(),  width=0.36, label="importance (soft)", color="#d6eaf8", edgecolor="#2874a6")
    ax.bar(xs + 0.18, load.numpy(), width=0.36, label="load (hard)",       color="#fdebd0", edgecolor="#b35900")
    ax.axhline(1.0 / E, ls="--", lw=0.8, color="grey", label=f"uniform = 1/E = {1/E:.2f}")
    ax.set_xticks(xs); ax.set_xticklabels([f"E{e}" for e in range(E)])
    ax.set_title(f"{title}  ->  aux = {aux_v.item():.2f}")
axes[0].set_ylabel("fraction")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# And our real (untrained) gate on the anchor tokens, recomputed with the same helper:
aux_real, imp_real, load_real = aux_loss(probs, idx[:, 0])
print("anchor tokens through the untrained gate:")
print(f"  importance per expert : {imp_real.numpy()}")
print(f"  load per expert       : {load_real.numpy()}")
print(f"  aux (recomputed)      : {aux_real.item():.4f}")
print(f"  aux (from the gate)   : {aux.item():.4f}   <- must match")


### How this gets used in real training

The MoE layer hands the fairness penalty back to the training loop, which adds a small slice of it on top of the normal "did you predict the next word?" loss:

```python
logits, ce_loss = model(x, y)              # the usual next-word loss
total_loss = ce_loss + 0.01 * fairness_penalty   # add a little fairness
total_loss.backward()
```

The `0.01` is a dial for *how much* we care about fairness:
- too small → the clinic collapses anyway (fairness ignored)
- too large → the receptionist obsesses over fairness and sends words to doctors that don't fit, hurting quality
- `0.01` is the well-tested middle ground.

### Why this particular recipe (simple to deep)

- **Simple:** the penalty is large when work is lopsided and small when it's even — exactly the nudge we want.
- **Practical:** multiplying confidence × patient-share means a doctor only gets penalized when they hog *both*. A doctor the receptionist *likes* but rarely actually picks (or vice-versa) isn't punished — only true hogs are.
- **Deeper:** because only the "confidence" half can be steered by training, the penalty works by quietly pulling confidence away from whichever doctors are already overloaded. Push it to the limit and the only way to make the penalty as small as possible is to give every doctor an equal share — which is the balanced clinic we wanted.


---
## 5.3 The Doctors (the Experts)

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#e8e8e8,stroke:#bbb,color:#777
    style aux fill:#e8e8e8,stroke:#bbb,color:#777
    style exp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style comb fill:#e8e8e8,stroke:#bbb,color:#777
    style hyb fill:#e8e8e8,stroke:#bbb,color:#777
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

### What we're doing here, and why

Here's the reassuring part: **a "doctor" is nothing new.** Each expert is literally the SwiGLU feed-forward network you already built in Part 3 — same math, the code just renames the pieces (`inp1/inp2/out` instead of `w1/w2/w3`).

$$\text{Expert}(x) = \big( (x W_{\text{inp1}}) \odot \text{SiLU}(x W_{\text{inp2}}) \big) W_{\text{out}}$$

The only change from Part 3 is that we keep **several copies** side by side (4 in our example) instead of one. They start with different random weights, and — once the receptionist is sending each one its own kind of word — they drift apart into specialists. Below we build a single doctor and pass all four words through it, just to confirm it behaves like the FFN you know.


In [ ]:
from experts import ExpertMLP

torch.manual_seed(1)
expert = ExpertMLP(dim=C, mult=4, swiglu=True)

with torch.no_grad():
    y_one = expert(x_flat)            # all 4 anchor tokens through ONE expert

print("input :", tuple(x_flat.shape))
print("output:", tuple(y_one.shape), "  (shape preserved, like every FFN)")
n_per_expert = sum(p.numel() for p in expert.parameters())
print(f"params per expert: {n_per_expert:,}   (formula 12*C^2 = {12*C*C:,})")


### The whole point: hire many, pay for few

This is the idea that makes MoE worth the trouble. Two different costs to keep separate:

- **How much the clinic knows** = how many doctors are *on staff* (all their weights live in memory).
- **What each patient costs** = how many doctors that patient actually *visits*.

A normal single-FFN layer is one doctor: it knows one doctor's worth of stuff, and every word pays for one doctor. MoE breaks those apart:

| | knowledge on staff | cost per word |
|---|---|---|
| single SwiGLU (Parts 1/3) | 1 doctor | 1 doctor |
| MoE (E doctors, visit k) | **E doctors** | **k doctors** |

So with 8 doctors where each word visits 2, the clinic *knows* 8× as much while each word only pays for 2. That's the real trick behind models like Mixtral: tons of total capacity, but only a small slice runs for any given word. Let's see the numbers for our tiny clinic:


In [ ]:
# Bar chart: parameters vs per-token compute, dense vs our toy MoE (E=4, k=2)
dense_params = 12 * C * C
moe_params   = E * dense_params + (C * E + E)        # experts + router
dense_flops  = dense_params                           # proportional
moe_flops    = K * dense_params + (C * E + E)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, (d, m), title in [
    (axes[0], (dense_params, moe_params), "Parameters (capacity)"),
    (axes[1], (dense_flops,  moe_flops),  "FLOPs per token (compute)"),
]:
    bars = ax.bar(["dense SwiGLU", f"MoE (E={E}, k={K})"], [d, m],
                  color=["#d6eaf8", "#d5f5e3"], edgecolor=["#2874a6", "#28a745"])
    for b, v in zip(bars, [d, m]):
        ax.text(b.get_x() + b.get_width()/2, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(title)
plt.tight_layout(); plt.show()

print(f"capacity ratio : {moe_params / dense_params:.2f}x   (~E = {E})")
print(f"compute ratio  : {moe_flops  / dense_flops:.2f}x   (~k = {K})")


### Shape trace

| Stage | Shape |
|---|---|
| tokens routed to expert e | `(S_e, C)` — S_e varies per expert |
| value branch `a = x @ inp1` | `(S_e, 4C)` |
| gate branch `b = SiLU(x @ inp2)` | `(S_e, 4C)` |
| output `(a*b) @ out` | `(S_e, C)` |

### Why do doctors end up specializing? (simple to deep)

They start nearly the same, so where does specialization come from?

- **Simple:** they begin with *different random weights*, so they give slightly different answers from word one — and the receptionist notices.
- **Practical:** small differences snowball. A doctor who's a touch better at, say, punctuation-heavy words gets sent more of them, practices on them, and pulls further ahead — a "the good get better at their thing" loop. (And remember 5.2: the fairness penalty is what stops this from snowballing into *one* doctor winning everything.)
- **Deeper:** the receptionist and doctors are trained *together* toward one goal. The cheapest way to do well is to hand each doctor a coherent slice of words and let them master it. Specialization isn't luck — it's the most efficient arrangement, so training drifts toward it.


---
## 5.4 The Full Visit (Dispatch and Combine)

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#e8e8e8,stroke:#bbb,color:#777
    style aux fill:#e8e8e8,stroke:#bbb,color:#777
    style exp fill:#e8e8e8,stroke:#bbb,color:#777
    style comb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style hyb fill:#e8e8e8,stroke:#bbb,color:#777
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

### What we're doing here, and why

Now we put it together. We have the receptionist's decisions (5.1) and the doctors (5.3). The visit itself is two steps:

1. **Dispatch** — send each word to its 2 chosen doctors and let them examine it.
2. **Combine** — merge the 2 doctors' opinions into one answer, weighting each by how strongly the receptionist recommended that doctor.

In plain terms, each word's final answer is *a confidence-weighted blend of its 2 doctors' opinions*:

$$\text{answer for a word} = (\text{weight}_1 \times \text{Doctor}_1\text{'s opinion}) + (\text{weight}_2 \times \text{Doctor}_2\text{'s opinion})$$

The code (`moe.py`) does this by going doctor-by-doctor: for each doctor it grabs just the words that chose them, runs them through, and adds the weighted result into the right rows:

```python
for e in range(n_expert):          # for each doctor
    for slot in range(k):          # for each of the word's 2 slots
        sel = (idx[:, slot] == e)  # which words picked this doctor here?
        if sel.any():
            y[sel] += w[sel, slot:slot+1] * experts[e](x_flat[sel])
```

It *looks* like a lot of looping, but each doctor only ever sees their own words — so the real work is just "every word visits 2 doctors," no matter how many doctors are on staff. (Two simplifications vs. a giant production system: we never turn a word away even if a doctor gets swamped, and the real systems spread doctors across many GPUs. Fine for learning.)

### Let's run our four words through the whole clinic


In [ ]:
from moe import MoE

torch.manual_seed(7)   # same seed as 5.1 -> the MoE's internal gate = same routing
moe = MoE(dim=C, n_expert=E, k=K, mult=4, swiglu=True)

with torch.no_grad():
    y_moe, aux_moe = moe(x_anchor)

print("input :", tuple(x_anchor.shape))
print("output:", tuple(y_moe.shape), "  (drop-in FFN: shape preserved)")
print(f"aux   : {aux_moe.item():.4f}")
print()

# Show the routing table this layer actually used
with torch.no_grad():
    idx_m, w_m, _ = moe.gate(x_flat)
print("Routing table:")
print(f"  {'token':9s} -> experts {'':6s} weights")
for t, tok in enumerate(TOKENS):
    print(f"  {tok:9s} -> {idx_m[t].tolist()!s:9s} {[round(v,3) for v in w_m[t].tolist()]}")


### Prove it by hand — rebuild the first word's answer ourselves

Let's check the clinic really does "weighted blend of 2 doctors." Take the word *I*. We'll call its 2 chosen doctors directly, blend their opinions by the receptionist's weights, and confirm we get the exact same row the full layer produced:

$$\text{answer}_I = (\text{weight}_1 \times \text{Doctor}_1(I)) + (\text{weight}_2 \times \text{Doctor}_2(I))$$


In [ ]:
x0 = x_flat[0]                                    # the word "I", shape (C,)
e_a, e_b = idx_m[0, 0].item(), idx_m[0, 1].item()
w_a, w_b = w_m[0, 0].item(),  w_m[0, 1].item()

with torch.no_grad():
    y0_manual = w_a * moe.experts[e_a](x0) + w_b * moe.experts[e_b](x0)

print(f"token 'I' routed to experts {e_a} (w={w_a:.3f}) and {e_b} (w={w_b:.3f})")
print()
print("y0 manual:", y0_manual.numpy())
print("y0 layer :", y_moe[0, 0].numpy())
print("max abs diff:", (y0_manual - y_moe[0, 0]).abs().max().item())


### Picture it — how busy is each doctor?

Four words is too few to see a pattern. Let's send **512 random words** through the receptionist and count how many picked each doctor as their top choice. A fresh, untrained receptionist spreads them *roughly* (not perfectly) evenly — the dashed line is the perfectly-fair share.


In [ ]:
torch.manual_seed(123)
x_big = torch.randn(512, C)
with torch.no_grad():
    idx_big, w_big, aux_big = moe.gate(x_big)

counts = torch.bincount(idx_big[:, 0], minlength=E)

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar([f"Expert {e}" for e in range(E)], counts.tolist(),
              color="#d5f5e3", edgecolor="#28a745")
ax.axhline(512 / E, ls="--", color="grey", lw=0.8, label=f"perfectly uniform = {512//E}")
for b, v in zip(bars, counts.tolist()):
    ax.text(b.get_x() + b.get_width()/2, v, str(v), ha="center", va="bottom")
ax.set_ylabel("# tokens (primary)"); ax.set_title(f"Primary-expert load, 512 random tokens  (aux = {aux_big.item():.3f})")
ax.legend(); plt.tight_layout(); plt.show()


### Does training actually reach everyone?

For the clinic to improve, training has to be able to adjust *both* the receptionist and every doctor. Let's do one backward pass and confirm a learning signal reaches each of them (a non-zero "grad" means "training can tune this part").


In [ ]:
# One backward pass: gradients must reach BOTH the router and the experts.
moe.zero_grad()
y_g, aux_g = moe(x_anchor)
loss = y_g.pow(2).mean() + 0.01 * aux_g     # fake task loss + weighted aux
loss.backward()

print(f"router  w_g.weight.grad   exists: {moe.gate.w_g.weight.grad is not None}, "
      f"norm = {moe.gate.w_g.weight.grad.norm():.4f}")
for e in range(E):
    g = moe.experts[e].inp1.weight.grad
    print(f"expert {e} inp1.weight.grad exists: {g is not None}, norm = {0.0 if g is None else g.norm():.4f}")


**What you see:** every doctor who saw at least one word gets a learning signal, and the receptionist gets one too — through two routes at once: the weights it used to blend opinions, and the fairness penalty from 5.2. Everything in the clinic learns together.

### Shape trace

| Stage | Shape |
|---|---|
| input `x` | `(B, T, C)` = `(1, 4, 8)` |
| `x_flat` | `(S, C)` = `(4, 8)` |
| `idx`, `w` | `(4, 2)` each |
| tokens for expert e | `(S_e, 8)` — varies |
| expert output | `(S_e, 8)` |
| `y` after combine | `(4, 8)` |
| reshaped output | `(1, 4, 8)` + scalar `aux` |


---
## 5.5 Adding a General Doctor (the Hybrid Block)

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style flat fill:#e8e8e8,stroke:#bbb,color:#777
    style gate fill:#e8e8e8,stroke:#bbb,color:#777
    style aux fill:#e8e8e8,stroke:#bbb,color:#777
    style exp fill:#e8e8e8,stroke:#bbb,color:#777
    style comb fill:#e8e8e8,stroke:#bbb,color:#777
    style hyb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style out fill:#e8e8e8,stroke:#bbb,color:#777
```

### What we're doing here, and why

The pure clinic has a rough early phase: the receptionist is still learning, so it sometimes sends a word to the wrong specialists and gets a poor answer. A simple safety net: **also keep one general doctor who sees every word**, and blend their steady opinion with the specialists' answer.

That's the hybrid block. One knob, `alpha`, sets the mix:

$$\text{answer} = \alpha \times (\text{general doctor}) + (1 - \alpha) \times (\text{specialist clinic})$$

- `alpha = 1` → only the general doctor (rock-steady, but less specialized knowledge)
- `alpha = 0` → only the specialist clinic (most knowledge, but noisier early on)
- in between → both at once. The always-on general doctor gives every word a reliable baseline answer while the specialists add their depth on top. (Some real models, like DeepSeek's, build in exactly this "always-on doctor" idea.)

### Let's dial alpha and check the two extremes behave


In [ ]:
from block_hybrid import HybridFFN

torch.manual_seed(3)
hyb = HybridFFN(dim=C, alpha=0.5, n_expert=E, k=K)

with torch.no_grad():
    # The module's own paths, computed separately:
    y_dense = hyb.dense(x_anchor)
    y_moe_p, _ = hyb.moe(x_anchor)

    print(f"{'alpha':>6s} {'|y - dense|':>12s} {'|y - moe|':>10s}")
    for alpha in [1.0, 0.5, 0.0]:
        hyb.alpha = alpha
        y_h, aux_h = hyb(x_anchor)
        d_dense = (y_h - y_dense).abs().max().item()
        d_moe   = (y_h - y_moe_p).abs().max().item()
        print(f"{alpha:6.1f} {d_dense:12.2e} {d_moe:10.2e}")


**What you see:** at `alpha = 1` the answer is exactly the general doctor's (difference ≈ 0); at `alpha = 0` it's exactly the specialist clinic's; at `0.5` it sits halfway between. The fairness penalty still comes from the specialist side unchanged — you keep adding it during training just like in 5.2.


---
## 5.6 Putting the Clinic Into a Real Block

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style flat fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style gate fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style aux fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style exp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style comb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style hyb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style out fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

### What we're doing here, and why

Everything so far was the feed-forward piece in isolation. The payoff is plugging it into a real Transformer block — the same one from Part 3 — by simply **swapping its single feed-forward for our clinic**. Nothing else about the block changes: same attention, same norms, same residual shortcuts. We build it below from the Part 3 + Part 5 pieces you already have:

```mermaid
graph TD
    inp["Input (B, T, C)"]
    rms1["RMSNorm 1 (Part 3)"]
    attn["Modern Attention (Part 3)<br>RoPE + GQA"]
    add1["+ residual"]
    rms2["RMSNorm 2 (Part 3)"]
    moe["MoE FFN (Part 5)<br>router + experts"]
    add2["+ residual"]
    outp["Output (B, T, C) + aux"]

    inp --> rms1 --> attn --> add1
    inp --> add1
    add1 --> rms2 --> moe --> add2
    add1 --> add2
    add2 --> outp

    style inp fill:#e8e8e8,stroke:#bbb,color:#777
    style rms1 fill:#d6eaf8,stroke:#2874a6,color:#000
    style attn fill:#d6eaf8,stroke:#2874a6,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style rms2 fill:#d6eaf8,stroke:#2874a6,color:#000
    style moe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style outp fill:#e8e8e8,stroke:#bbb,color:#777
```

Blue = the Part 3 pieces you already built, green = today's clinic dropped into the feed-forward slot. The only real change from a normal Part 3 block: this block also hands back the **fairness penalty**, which the training loop adds on top of the usual next-word loss. Everything else — attention, the residual "+" shortcuts, the norms — is untouched.


In [ ]:
from rmsnorm import RMSNorm
from attn_modern import CausalSelfAttentionModern

class MoEBlockModern(nn.Module):
    """Part 3 modern block with the dense SwiGLU swapped for a Part 5 MoE."""
    def __init__(self, n_embd, n_head, n_kv_head=None, n_expert=4, k=2):
        super().__init__()
        self.ln1  = RMSNorm(n_embd)
        self.attn = CausalSelfAttentionModern(n_embd, n_head, rope=True,
                                              max_pos=64, n_kv_head=n_kv_head)
        self.ln2  = RMSNorm(n_embd)
        self.ffn  = MoE(dim=n_embd, n_expert=n_expert, k=k)   # <- the swap

    def forward(self, x, kv_cache=None, start_pos=0):
        a, kv_cache = self.attn(self.ln1(x), kv_cache=kv_cache, start_pos=start_pos)
        x = x + a                                  # attention residual
        m, aux = self.ffn(self.ln2(x))             # MoE returns (y, aux)
        x = x + m                                  # FFN residual
        return x, kv_cache, aux

torch.manual_seed(0)
block = MoEBlockModern(n_embd=C, n_head=4, n_kv_head=2, n_expert=E, k=K)

y_blk, kv, aux_blk = block(x_anchor)
print("input :", tuple(x_anchor.shape))
print("output:", tuple(y_blk.shape), "  (stackable: shape preserved)")
print(f"aux   : {aux_blk.item():.4f}")
print(f"KV cache: k={tuple(kv.k.shape)}  (GQA: only n_kv_head=2 KV heads, as in Part 3)")


In [ ]:
# End-to-end gradient check through the WHOLE block, with the aux loss attached
# exactly the way a training loop would do it:
block.zero_grad()
y2, _, aux2 = block(x_anchor)
total_loss = y2.pow(2).mean() + 0.01 * aux2     # stand-in for CE + lambda*aux
total_loss.backward()

named = {
    "attention wq":   block.attn.wq.weight.grad,
    "router w_g":     block.ffn.gate.w_g.weight.grad,
    "expert 0 inp1":  block.ffn.experts[0].inp1.weight.grad,
    "rmsnorm 2 gain": block.ln2.weight.grad,
}
for name, g in named.items():
    print(f"  grad through {name:15s}: {'OK' if g is not None and g.norm() > 0 else 'missing'}  (norm={0.0 if g is None else g.norm():.4f})")


### What a real, fully-trained clinic adds (beyond this part)

Our version keeps things simple for learning. Production MoE models add a few practical extras — here they are in plain terms, so the names don't surprise you later:

| Extra | In clinic terms | Why |
|---|---|---|
| capacity factor | cap how many patients one doctor can take; turn away the overflow | keeps every doctor's workload an even, hardware-friendly size |
| expert parallelism | put each doctor in a different building (GPU) | lets you have *way* more doctors than fit on one machine |
| shared experts | keep a general doctor always on (5.5, done properly) | a reliable baseline answer for every word |
| router z-loss | tell the receptionist not to shout (keep its scores from blowing up) | stops numerical blow-ups during training |
| alternating layers | only some floors of the building are clinics; others are single doctors | balances cost and quality across the whole model |


---
## Closing the loop

```mermaid
graph TD
    input["Input x (B, T, C)<br>attention output, after RMSNorm 2"]
    flat["flatten to (S, C)<br>S = B*T tokens"]
    gate["5.1 TopKGate router<br>Linear(C to E) -> softmax -> top-k"]
    aux["5.2 Load-balancing aux loss<br>E * sum(importance * load)"]
    exp["5.3 Experts<br>E x SwiGLU MLPs"]
    comb["5.4 Dispatch / Combine<br>y[sel] += w * Expert_e(x[sel])"]
    hyb["5.5 Hybrid blend (optional)<br>alpha*Dense + (1-alpha)*MoE"]
    out["Output y (B, T, C) + aux<br>goes into + residual"]

    input --> flat --> gate
    gate --> comb
    gate -.-> aux
    exp --> comb
    comb --> hyb --> out
    aux -.-> out

    style input fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style flat fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style gate fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style aux fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style exp fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style comb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style hyb fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style out fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
```

You now have everything needed to read every file in `part_5/`:

- [gating.py](gating.py) — sections 5.1 (router) and 5.2 (aux loss)
- [experts.py](experts.py) — section 5.3
- [moe.py](moe.py) — section 5.4
- [block_hybrid.py](block_hybrid.py) — section 5.5
- [demo_moe.py](demo_moe.py) — the load-histogram demo (same as our 512-token plot)
- [README.md](README.md) — compact theory + distributed-training notes

**The one-sentence takeaway:** MoE swaps the single feed-forward (one overworked general doctor) for *a team of specialists plus a receptionist*, so the model can know far more without each word costing more — and the only price is a bit of bookkeeping (the fairness penalty) to keep one doctor from hogging all the patients.

### Quick recap of the whole clinic

- **5.1 Receptionist** — scores the doctors for each word, picks the best 2.
- **5.2 Fairness** — a small penalty that stops one doctor from taking everyone.
- **5.3 Doctors** — each is just a Part 3 SwiGLU; many on staff, each specializes.
- **5.4 The visit** — send each word to its 2 doctors, blend their answers by confidence.
- **5.5 General doctor** — optionally mix in one always-on doctor for stability.
- **5.6 Real block** — drop the clinic into a Part 3 Transformer block in place of the feed-forward.

### What's next

Part 6 begins the *alignment* arc: **Supervised Fine-Tuning** (SFT) — taking the Part 4 base checkpoint and teaching it to follow a prompt/response format with masked labels (only the response is supervised). Parts 6-9 all require the Part 4 BPE tokenizer + base checkpoint (`cd part_4 && python orchestrator.py --demo` if you haven't trained one yet).
